# Gelombang Guru Sekeluarga (same-family teacher)

> **Notebook ini diturunkan otomatis** dari `finetune-gemma-energy.ipynb` melalui
> `scripts/make_samefam_notebooks.py`. Jangan menyunting hyperparameter di sini.
> Bila resep pelatihan perlu berubah, ubah notebook asalnya lalu bangkitkan
> ulang berkas ini agar kedua gelombang tetap identik.

**Perbedaan satu-satunya terhadap notebook asal:**

| Aspek | Gelombang 1 (asal) | Gelombang 2 (notebook ini) |
|---|---|---|
| Guru pembangkit dataset | DeepSeek (lintas keluarga) | `google/gemma-4-31b-it` (sekeluarga) |
| Direktori dataset Kaggle | `finetune` | `finetune-teacher-gemma` |
| Nama artefak keluaran | tanpa sufiks | bersufiks `-fam` |

Seluruh hyperparameter QLoRA, base model, chat template, seed, dan jalur ekspor
GGUF **tidak berubah**, sehingga perbedaan kualitas narasi yang teramati dapat
diatribusikan pada asal guru, bukan pada perlakuan pelatihan.

**Wajib dicatat setelah run:** SHA256 kedua berkas dataset, tanggal run,
penyedia hulu OpenRouter yang melayani guru, dan tag Ollama final.


## 1. Setup & Cek GPU

Verifikasi T4 aktif sebelum instalasi.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Upload Dataset

Jalankan sel di bawah, lalu upload **train.jsonl** dan **val.jsonl** dari komputermu.

In [1]:
import os
import json

# Sesuaikan 'nama-dataset-kamu' dengan nama direktori dataset yang di-add ke Kaggle.
# Jika file berada di direktori output (working), ubah ke '/kaggle/working/'
dataset_dir = "/kaggle/input/datasets/yogafsyahputra/finetune-teacher-gemma" 

train_path = os.path.join(dataset_dir, "train.jsonl")
val_path = os.path.join(dataset_dir, "val.jsonl")

print("Mengecek file dataset di Kaggle...")
assert os.path.exists(train_path), f"File tidak ditemukan di: {train_path}"
assert os.path.exists(val_path), f"File tidak ditemukan di: {val_path}"
print("Dataset ditemukan!")

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_data = load_jsonl(train_path)
val_data = load_jsonl(val_path)

print(f"train: {len(train_data)} contoh")
print(f"val:   {len(val_data)} contoh")

print("\n=== CONTOH ===")
print("INSTRUCTION:", train_data[0]["instruction"][:120], "...")
print("\nINPUT:\n", train_data[0]["input"][:200])
print("\nOUTPUT:\n", train_data[0]["output"][:300])

Mengecek file dataset di Kaggle...
Dataset ditemukan!
train: 194 contoh
val:   49 contoh

=== CONTOH ===
INSTRUCTION: Ubah statistik deterministik sebuah grafik konsumsi listrik rumah tangga menjadi narasi insight 3-5 kalimat dalam Bahasa ...

INPUT:
 chart_id: sub_metering_comparison
record_count: 2033491
avg_sub_metering_1_wh: 0,9635
avg_sub_metering_2_wh: 1,7508
avg_sub_metering_3_wh: 7,1857
total_sub_metering_1_wh: 1959268,6
total_sub_metering_

OUTPUT:
 Berdasarkan data konsumsi listrik rumah tangga, sub-metering ketiga mencatat rata-rata energi tertinggi sebesar 7,1857 Wh, jauh melampaui sub-metering pertama (0,9635 Wh) dan kedua (1,7508 Wh). Disparitas signifikan ini mengindikasikan bahwa beban dasar rumah tangga didominasi oleh peralatan yang te


## 3. Load Model gemma2:2b 4-bit

Unsloth memuat base model terkuantisasi 4-bit — muat nyaman di T4 16GB, dan konsisten dengan model produksimu (gemma2:2b GGUF Q4 di Ollama).

In [ ]:
%%capture
# 1. Instalasi Unsloth jalur khusus Kaggle
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Downgrade ke versi stabil untuk Gemma-2 beserta seluruh dependensi pendukungnya
!pip install --no-deps "transformers==4.55.4" "trl==0.21.0" "peft==0.14.0" "tokenizers==0.21.2" "huggingface-hub<1.0"

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024   # cukup untuk statistik + narasi
dtype = None            # auto: float16 untuk T4
load_in_4bit = True     # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model & tokenizer dimuat.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-06-26 07:17:53.929783: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782458274.411714     182 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782458274.529161     182 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782458275.597235     182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782458275.597292     182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782458275.597295     182 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Gemma2 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

Model & tokenizer dimuat.


### Sanity Check — Uji Base Model SEBELUM Training


In [3]:
# Verifikasi versi (harus transformers 4.55.x, BUKAN 5.x)
import transformers, trl
print("transformers:", transformers.__version__, "| trl:", trl.__version__)
assert transformers.__version__.startswith("4."), "STOP: transformers 5.x memecah gemma2. Restart & cek instalasi."

# Sanity check: base model harus bisa bahasa waras sebelum di-train
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"Jelaskan dalam satu kalimat apa itu konsumsi listrik."}]
ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=60, temperature=0.3, do_sample=True)
print("\n=== OUTPUT BASE MODEL ===")
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
print("\n>>> Jika output di atas WARAS (kalimat Indonesia normal), lanjut training.")
print(">>> Jika ACAK/gibberish, STOP — masalah versi, jangan lanjut.")
FastLanguageModel.for_training(model)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


transformers: 4.55.4 | trl: 0.21.0

=== OUTPUT BASE MODEL ===
Konsumsi listrik adalah jumlah energi listrik yang digunakan dalam suatu periode waktu, biasanya diukur dalam watt-jam (Wh) atau kilowatt-jam (kWh). 


>>> Jika output di atas WARAS (kalimat Indonesia normal), lanjut training.
>>> Jika ACAK/gibberish, STOP — masalah versi, jangan lanjut.


Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): GemmaFixedRotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layerno

In [ ]:
!pip install --upgrade peft

In [4]:
# Pasang LoRA adapter (parameter sesuai panduan: tugas sempit, hindari overfit)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # rank LoRA
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,         # sedikit dropout untuk regularisasi
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # hemat VRAM
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

# VERIFIKASI KRITIS: pastikan LoRA benar-benar menempel ke modul.
# Bug Transformers 5.x menyebabkan "0 QKV/O/MLP layers" -> training sia-sia & output acak.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}%)")
assert trainable > 1_000_000, (
    "STOP: LoRA TIDAK menempel (trainable params terlalu kecil). "
    "Ini bug versi Transformers. Cek log: jika 'patched ... 0 QKV layers', "
    "berarti versi salah. Restart & pastikan transformers 4.55.x."
)
print("LoRA adapter terpasang dengan benar (>1M trainable params).")

Unsloth 2026.6.9 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


Trainable params: 20,766,720 (1.28%)
LoRA adapter terpasang dengan benar (>1M trainable params).


## 4. Format Data dengan Chat Template

Gemma-2 memakai format chat tertentu. Kita gabung `instruction` + `input` sebagai pesan user, dan `output` sebagai jawaban model.

In [5]:
from datasets import Dataset

# Gemma-2 chat template
def to_chat(example):
    user_msg = example["instruction"].strip() + "\n\n=== STATISTIK INPUT ===\n" + example["input"].strip()
    messages = [
        {"role":"user","content":user_msg},
        {"role":"assistant","content":example["output"].strip()},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = Dataset.from_list(train_data).map(to_chat)
val_ds   = Dataset.from_list(val_data).map(to_chat)

print("=== CONTOH TEKS TERFORMAT ===")
print(train_ds[0]["text"][:600])

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

=== CONTOH TEKS TERFORMAT ===
<bos><start_of_turn>user
Ubah statistik deterministik sebuah grafik konsumsi listrik rumah tangga menjadi narasi insight 3-5 kalimat dalam Bahasa Indonesia formal-teknis. Setiap angka harus berasal dari statistik input (atau turunannya seperti load factor dan koefisien variasi) dan disertai satuan baku; gunakan minimal dua istilah domain yang relevan tanpa mengarang angka.

=== STATISTIK INPUT ===
chart_id: sub_metering_comparison
record_count: 2033491
avg_sub_metering_1_wh: 0,9635
avg_sub_metering_2_wh: 1,7508
avg_sub_metering_3_wh: 7,1857
total_sub_metering_1_wh: 1959268,6
total_sub_metering


## 5. Training QLoRA

Konfigurasi: 3 epoch, effective batch 8 (batch 2 × grad accum 4), LR 2e-4. Untuk ~194 contoh, perkiraan waktu **belasan menit hingga ~30 menit** di T4.

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        # PENTING: matikan penyimpanan checkpoint otomatis.
        # Versi Unsloth+TRL terbaru gagal pickle SFTConfig saat _save_checkpoint.
        # Dataset kecil tidak butuh checkpoint per-epoch; adapter disimpan manual di Bagian 7.
        save_strategy = "no",
        eval_strategy = "epoch",   # tetap lihat validation loss tiap epoch
        report_to = "none",
    ),
)

# === KRUSIAL: latih HANYA pada bagian jawaban (narasi), bukan bagian input ===
# Tanpa ini, model belajar memproduksi blok statistik input juga, lalu saat
# inference malah memuntahkan "=== STATISTIK INPUT ===" / statistik baru.
# Masking ini membuat loss hanya dihitung pada respons assistant.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)
print("✅ train_on_responses_only aktif — loss hanya pada bagian narasi.")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/194 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/49 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/194 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/194 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/49 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/49 [00:00<?, ? examples/s]

✅ train_on_responses_only aktif — loss hanya pada bagian narasi.


### Verifikasi Masking — pastikan hanya narasi yang dilatih


In [7]:
# Cek satu contoh: bagian yang TIDAK di-mask (yang benar-benar dilatih)
import numpy as np
ex = trainer.train_dataset[0]
labels = np.array(ex["labels"])
input_ids = np.array(ex["input_ids"])

# Token yang dilatih = label != -100
trained_ids = input_ids[labels != -100]
masked_count = int((labels == -100).sum())
trained_count = int((labels != -100).sum())

print(f"Token di-mask (tidak dilatih / bagian input): {masked_count}")
print(f"Token dilatih (bagian narasi): {trained_count}")
print("\n=== TEKS YANG DILATIH (harus HANYA narasi, bukan statistik input) ===")
print(tokenizer.decode(trained_ids, skip_special_tokens=False)[:600])
print("\n>>> Jika di atas hanya narasi Bahasa Indonesia, masking BENAR.")
print(">>> Jika muncul 'chart_id:' / 'STATISTIK INPUT' / angka statistik, masking SALAH.")

Token di-mask (tidak dilatih / bagian input): 261
Token dilatih (bagian narasi): 172

=== TEKS YANG DILATIH (harus HANYA narasi, bukan statistik input) ===
Berdasarkan data konsumsi listrik rumah tangga, sub-metering ketiga mencatat rata-rata energi tertinggi sebesar 7,1857 Wh, jauh melampaui sub-metering pertama (0,9635 Wh) dan kedua (1,7508 Wh). Disparitas signifikan ini mengindikasikan bahwa beban dasar rumah tangga didominasi oleh peralatan yang terhubung ke sub-metering ketiga, sementara sub-metering lainnya berkontribusi sebagai beban rendah yang lebih fluktuatif. Pola tersebut konsisten dengan profil beban residensial tipikal, di mana perangkat dengan daya tinggi seperti pemanas atau pompa menjadi kontributor utama energi total. Temuan ini

>>> Jika di atas hanya narasi Bahasa Indonesia, masking BENAR.
>>> Jika muncul 'chart_id:' / 'STATISTIK INPUT' / angka statistik, masking SALAH.


In [8]:
# Statistik memori sebelum training
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved()/1024/1024/1024, 2)
max_mem = round(gpu_stats.total_memory/1024/1024/1024, 2)
print(f"GPU: {gpu_stats.name}, total {max_mem} GB")
print(f"Terpakai sebelum training: {start_mem} GB")

GPU: Tesla T4, total 14.56 GB
Terpakai sebelum training: 2.49 GB


In [9]:
# Training. Jika muncul PicklingError pada langkah penyimpanan checkpoint,
# itu bug versi Unsloth+TRL, BUKAN kegagalan training. Kita sudah set
# save_strategy="no" di atas untuk menghindarinya. Adapter disimpan manual di Bagian 7.
trainer_stats = trainer.train()
print("\n✅ Training selesai. Loss akhir:", trainer_stats.metrics.get("train_loss"))


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 194 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Epoch,Training Loss,Validation Loss
1,1.376000,0.953086
2,0.742600,0.742135
3,0.640700,0.692737


Unsloth: Not an error, but Gemma2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



✅ Training selesai. Loss akhir: 1.0281551006512764


In [10]:
# Ringkas penggunaan memori & waktu
used_mem = round(torch.cuda.max_memory_reserved()/1024/1024/1024, 2)
print(f"Waktu training: {trainer_stats.metrics['train_runtime']:.0f} detik")
print(f"Puncak VRAM: {used_mem} GB / {max_mem} GB")

Waktu training: 282 detik
Puncak VRAM: 8.93 GB / 14.56 GB


## 6. Uji Inferensi — Sanity Check Anti-Halusinasi

Uji dengan statistik nyata. **Periksa manual:** apakah setiap angka di output berasal dari input? Apakah jargon dipakai tepat? Tidak ada benchmark/p-value karangan?

In [11]:
FastLanguageModel.for_inference(model)  # mode inferensi 2x lebih cepat

test_input = '''chart_id: hourly_consumption_pattern
hours: 24
avg_global_active_power_kw: 1,091
min_hour: 04:00
min_value_kw: 0,444
max_hour: 20:00
max_value_kw: 1,899
unit: kW'''

instruction = train_data[0]["instruction"]  # pakai instruction yang sama
user_msg = instruction + "\n\n=== STATISTIK INPUT ===\n" + test_input

messages = [{"role":"user","content":user_msg}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 256,
    temperature = 0.4,
    top_p = 0.9,
    do_sample = True,
)
result = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== NARASI HASIL FINE-TUNE ===\n")
print(result)

=== NARASI HASIL FINE-TUNE ===

Profil beban harian menunjukkan beban puncak sebesar 1,899 kW pada pukul 20:00, sedangkan beban terendah tercatat sebesar 0,444 kW pada pukul 04:00. Load factor yang dihitung dari rasio rata-rata terhadap puncak adalah 0,537, mengindikasikan disparitas puncak-lembah yang cukup lebar. Pola ini konsisten dengan profil beban residensial yang umumnya mengalami lonjakan konsumsi pada malam hari. Disparitas tersebut berpotensi mendukung strategi load shifting untuk meratakan kurva beban.


**Checklist verifikasi manual:**
- [ ] Semua angka (1,091 / 0,444 / 1,899 kW) muncul benar, tidak ada angka baru
- [ ] Satuan kW disebut
- [ ] Minimal 2 istilah domain (load factor, base load, kurva beban, dll)
- [ ] Tidak ada benchmark eksternal / p-value / prediksi kuantitatif karangan
- [ ] Bahasa Indonesia formal-teknis, mengalir

Jika lolos, lanjut simpan. Jika model mengarang angka, pertimbangkan menurunkan `temperature` atau menambah contoh dataset.

## 7. Simpan Adapter & Export GGUF Q4 untuk Ollama

### 7a. Simpan adapter LoRA (cadangan ringan)

In [12]:
# Simpan adapter LoRA (kecil, ~50-100 MB). save_method eksplisit menghindari
# jalur penyimpanan TrainingArguments yang bermasalah.
model.save_pretrained("gemma2-energy-insight-fam-lora")
tokenizer.save_pretrained("gemma2-energy-insight-fam-lora")
print("✅ Adapter LoRA tersimpan di gemma2-energy-insight-fam-lora/")


✅ Adapter LoRA tersimpan di gemma2-energy-insight-lora/


### 7b. Export ke GGUF Q4 (untuk Ollama lokal di GTX 1650)

Ini meng-merge adapter ke base model lalu kuantisasi ke Q4_K_M. Hasil ~1.8 GB, muat di VRAM 4 GB.

> Proses ini agak lama (beberapa menit) karena mengompilasi llama.cpp.

In [13]:
# Export GGUF Q4_K_M untuk Ollama. Proses ini mengompilasi llama.cpp (beberapa menit).
# Jika gagal di tahap kompilasi, jalankan ulang sel ini sekali lagi (sering berhasil
# pada percobaan kedua setelah cache terbangun).
model.save_pretrained_gguf(
    "gemma2-energy-insight-fam-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("✅ GGUF Q4 tersimpan di gemma2-energy-insight-fam-gguf/")


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/5.23G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:25<00:00, 25.24s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/gemma2-energy-insight-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9789-mix-1f1aaa4 (app-b9789-mix-1f1aaa4-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gemma2-energy-insight-gguf_gguf/gemma-2-2b-it.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['gemma2-energy-insight-gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model gemma2-energy-insight-gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to gemma2-energy-insight-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f gemma2-energy-insight-gguf_gguf/Modelfile
✅ GGUF Q4 tersimpan di gemma2-energy-insight-gguf/


In [14]:
# Lihat file GGUF yang dihasilkan
import os
for root, dirs, fnames in os.walk("gemma2-energy-insight-fam-gguf"):
    for fn in fnames:
        p = os.path.join(root, fn)
        sz = os.path.getsize(p)/1024/1024
        print(f"{p}  ({sz:.0f} MB)")

gemma2-energy-insight-gguf/tokenizer.model  (4 MB)
gemma2-energy-insight-gguf/generation_config.json  (0 MB)
gemma2-energy-insight-gguf/model.safetensors  (4986 MB)
gemma2-energy-insight-gguf/tokenizer_config.json  (0 MB)
gemma2-energy-insight-gguf/chat_template.jinja  (0 MB)
gemma2-energy-insight-gguf/tokenizer.json  (33 MB)
gemma2-energy-insight-gguf/special_tokens_map.json  (0 MB)
gemma2-energy-insight-gguf/config.json  (0 MB)
gemma2-energy-insight-gguf/.cache/huggingface/.gitignore  (0 MB)
gemma2-energy-insight-gguf/.cache/huggingface/download/model.safetensors.metadata  (0 MB)


### 7c. Download GGUF ke komputer

File GGUF cukup besar. Download lewat sel ini, atau (lebih andal) mount Google Drive dan salin ke sana.

In [15]:
import glob
from IPython.display import FileLink

# 1. Sesuaikan path sesuai dengan lokasi folder yang terlihat di panel Output
# Berdasarkan screenshot, folder targetnya adalah: gemma2-energy-insight-fam-gguf_gguf
file_path = "gemma2-energy-insight-fam-gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf"

# 2. Cek apakah file ada
import os
if os.path.exists(file_path):
    print("File ditemukan! Klik link di bawah untuk mengunduh:")
    display(FileLink(file_path))
else:
    # Jika path di atas salah, kita cari otomatis di semua folder
    found_files = glob.glob("/kaggle/working/**/*.gguf", recursive=True)
    if found_files:
        print("File ditemukan di lokasi lain. Klik link di bawah:")
        for f in found_files:
            display(FileLink(f))
    else:
        print("File .gguf tidak ditemukan. Pastikan nama folder dan file sudah benar.")

File ditemukan! Klik link di bawah untuk mengunduh:


/kaggle/working/gemma2-energy-insight-gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf

## 8. Deploy di Ollama Lokal (jalankan di GTX 1650, bukan di Colab)

Setelah GGUF ada di komputermu, buat `Modelfile`:

```
FROM ./gemma2-energy-insight-fam-q4_k_m.gguf
PARAMETER temperature 0.4
PARAMETER top_p 0.9
PARAMETER num_gpu 99
SYSTEM """Anda adalah analis data energi yang menyusun narasi insight dari statistik konsumsi listrik rumah tangga. Setiap angka harus berasal dari statistik input dan disertai satuan baku. Gunakan istilah domain yang relevan tanpa mengarang angka, benchmark, atau prediksi."""
```

Lalu:
```bash
ollama create gemma2-energy-insight -f Modelfile
ollama run gemma2-energy-insight
```

**Integrasi ke proyek:** ganti backend Insight Agent agar memakai model `gemma2-energy-insight`. SQL/Repair/Reporter Agent tetap pakai `gemma2:2b` base — kamu hanya menukar model untuk satu peran.

> Catatan VRAM: dua model di 4GB bisa ketat. Karena alurmu sekuensial, set `keep_alive` pendek agar Ollama load/unload sesuai kebutuhan.

## 9. Evaluasi (untuk Bab 4 tesis)

Jalankan checker deterministik proyekmu pada laporan yang dihasilkan model base vs fine-tuned, bandingkan:
- Unit rule compliance (target: dari 0/4 naik)
- Numeric fact coverage (target: dari 0/2 naik)
- Tambah rubrik manual semantik (skala 1-5) pada ~20-30 sampel

Jangan over-claim: fine-tuning memperbaiki konsistensi satuan & jargon, tapi kedalaman interpretasi tetap dibatasi kapasitas model 2B.

# Fine-Tuning gemma2:2b untuk Insight Agent Domain Energi (QLoRA + Unsloth)

Proyek: `local-agentic-analytics` — Tugas Akhir Yoga Firman Syahputra

**Tujuan:** Melatih adapter QLoRA agar Insight Agent menghasilkan narasi konsumsi listrik yang kaya jargon statistika/kelistrikan **tanpa mengarang angka**.

**Lingkungan:** Google Colab, runtime **T4 GPU** (gratis). Pastikan: menu *Runtime → Change runtime type → T4 GPU*.

**Alur notebook:**
1. Setup & cek GPU
2. Upload dataset (train.jsonl, val.jsonl)
3. Load gemma2:2b 4-bit via Unsloth
4. Format data (chat template)
5. Training QLoRA
6. Uji inferensi cepat (sanity check anti-halusinasi)
7. Simpan adapter + export GGUF Q4 untuk Ollama lokal

> Catatan tesis: training di Colab tidak melanggar premis *local inference*. Yang lokal adalah deployment adapter hasil training pada GTX 1650.